# 07 — Entregas finais: RAIS espacializada (GeoPackage e Excel) e saldo anual de vínculos

**Objetivo:** gerar, em `04_entrega`, os três arquivos finais (mesmo padrão da entrega da espacialização de CNPJs):

1. **`rais_estab_espacial.gpkg`** (camada `rais_estab_espacial`, CRS EPSG:4674): **todos** os registros da RAIS (todos os anos), com as colunas originais da RAIS + a situação espacial. Registros sem ponto entram **sem geometria** e com `lat`/`lon` vazios.
2. **`rais_estab_espacial.xlsx`**: as mesmas linhas e colunas, **sem geometria**, uma aba por ano + aba `legenda`. `lat` e `lon` vazios nos registros sem ponto.
3. **`saldo_vinculos_ativos.xlsx`**: saldo **anual** de vínculos ativos (31/12), somando **todos** os registros da RAIS, inclusive os que não foi possível espacializar (que aparecem como `nao_localizavel`). Abas `por_ano`, `por_ano_confianca` e `legenda`.

**Colunas que a entrega acrescenta às da RAIS:**
- `linha_rais`: posição do registro na RAIS consolidada (rastreio).
- `situacao_espacial`: `espacializado` (tem ponto) ou `nao_localizavel` (não tem). O motivo detalhado fica em `motivo_cep` (coluna técnica).
- `classe_cep`, `motivo_cep`, `processo_cep`, `confianca_cep` (`alta`, `media`, `baixa`, `manual`), `ponto_metodo`, `n_partes`, `lat`, `lon`.

**Regras:** CEP de outro município **não** recebe rótulo próprio na situação espacial: o registro consta como de São José dos Campos (o CEP pode ter sido cadastrado errado) e fica `nao_localizavel`. O saldo soma todos os `qtd_vinc_ativos`.

**Entradas:** `rais_estab_sjc_classificada.parquet` e `ceps_espacializados.gpkg` (etapa 05) e `rais_estab_sjc_consolidado.parquet` (etapa 01, só para conferir que nada se perdeu).
**Estrutura:** 1 Preparação · 2 Funções · 3 Carregar entradas · 4 Montar a tabela final · 5 GeoPackage · 6 Excel dos registros · 7 Saldo anual · 8 Resumo

## 1. Preparação e configuração

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "07_exportar_entregas"                        # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
import time                                           # para medir o tempo de cada parte
import numpy as np                                    # contas com colunas
import pandas as pd                                   # tabelas
import pyogrio                                        # leitura/escrita de arquivos geográficos
import geopandas as gpd                               # tabelas com geometria

# --- Entradas (config.yaml) ---------------------------------------------------------------------------
ARQ_RAIS_CLASS = cfg.caminho("processamento.rais_classificada")        # etapa 05: rais + classe do CEP
ARQ_PONTOS_CEP = cfg.caminho("processamento.ceps_espacializados")      # etapa 05: um ponto por CEP (camada ceps_pontos)
ARQ_RAIS_CONSOL = cfg.caminho("processamento.rais_consolidada")        # etapa 01: rais original consolidada (só para conferir)

# --- Saídas (config.yaml) ------------------------------------------------------------------------------------
ARQ_GPKG = cfg.caminho("entrega.rais_espacial")                        # 04_entrega/rais_estab_espacial.gpkg
CAMADA_GPKG = cfg.camada("entrega.rais_espacial")                      # camada única do GeoPackage
ARQ_XLSX = cfg.caminho("entrega.rais_espacial_excel")                  # 04_entrega/rais_estab_espacial.xlsx
ARQ_SALDO = cfg.caminho("entrega.saldo_vinculos")                      # 04_entrega/saldo_vinculos_ativos.xlsx
ARQ_ANTIGO = cfg.caminho("entrega.rais_pontos_antigo")                 # camada de pontos antiga (substituída por esta etapa)

# --- Parâmetros ------------------------------------------------------------------------------------------------
CRS_METRICO = cfg.projeto("crs_metrico")                               # CRS dos pontos gravados na etapa 05
CRS_GRAUS = cfg.projeto("crs_graus")                                   # CRS da entrega (lat/lon em graus), como na entrega de CNPJs
CLASSES_PONTO = ["A", "B", "G", "P"]                                   # classes de CEP que têm ponto
COLUNAS_EXCLUIR = cfg.parametro("pontos.colunas_excluir")              # colunas da rais que ficam de fora da entrega
LIMITE_ALERTA = cfg.parametro("consolidacao.limite_variacao_alerta")   # alerta se os estabelecimentos variarem mais que isso de um ano para o outro
ORDEM_CONFIANCA = ["alta", "media", "baixa", "manual", "nao_localizavel"]   # ordem das linhas na aba por_ano_confianca

log.info("Configuração carregada!")
for _nome, _arq in [("rais classificada", ARQ_RAIS_CLASS), ("Pontos dos CEPs", ARQ_PONTOS_CEP), ("rais consolidada", ARQ_RAIS_CONSOL)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())
ARQ_GPKG.parent.mkdir(parents=True, exist_ok=True)                     # garante a pasta 04_entrega
if ARQ_ANTIGO.exists():
    log.warning("O arquivo antigo %s não é mais gerado (foi substituído por %s). Pode apagá-lo.", ARQ_ANTIGO.name, ARQ_GPKG.name)

## 2. Funções auxiliares

In [ ]:
from src.validacao import validar_tabela              # bloco padrão de validação (nulos, duplicados, perda)
from src.geo import para_gpkg                         # tipos compatíveis com GeoPackage
from src.planilha import formatar_excel               # formatação de abas (usada se o xlsxwriter não estiver instalado)

try:                                                  # o xlsxwriter grava Excel grande bem mais rápido que o openpyxl
    import xlsxwriter                                 # noqa: F401
    MOTOR_EXCEL = "xlsxwriter"
except ImportError:
    MOTOR_EXCEL = "openpyxl"                          # alternativa (mais lenta)
log.info("Motor do Excel: %s", MOTOR_EXCEL)

CARACTERES_INVALIDOS = r"[\x00-\x08\x0b\x0c\x0e-\x1f]"  # caracteres de controle que o Excel não aceita


def limpar_texto_excel(tabela):
    """Remove caracteres de controle (que o Excel recusa) das colunas de texto e devolve a tabela e quantas células mudaram."""
    out = tabela.copy()
    mudadas = 0
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]) or pd.api.types.is_datetime64_any_dtype(out[col]):
            continue                                                         # só colunas de texto
        como_texto = out[col].astype("string")
        tem = como_texto.str.contains(CARACTERES_INVALIDOS, regex=True, na=False)
        if tem.any():
            mudadas += int(tem.sum())
            out[col] = out[col].astype(object)                               # garante que aceita o texto limpo
            out.loc[tem, col] = como_texto[tem].str.replace(CARACTERES_INVALIDOS, "", regex=True).astype(object)
    return out, mudadas


def gravar_aba(escritor, nome_aba, tabela):
    """Grava uma tabela numa aba e aplica o padrão: cabeçalho em negrito, filtro, linha 1 congelada e larguras."""
    tabela.to_excel(escritor, sheet_name=nome_aba, index=False)
    if MOTOR_EXCEL == "xlsxwriter":
        aba = escritor.sheets[nome_aba]
        formato = escritor.book.add_format({"bold": True, "bg_color": "#DDEBF7"})
        for j, col in enumerate(tabela.columns):                           # reescreve o cabeçalho com o formato do padrão
            aba.write(0, j, col, formato)
            amostra = tabela[col].head(200).astype(str).str.len().max() if len(tabela) else 0
            larg = max(len(str(col)), 0 if pd.isna(amostra) else int(amostra)) + 2
            aba.set_column(j, j, min(max(larg, 8), 60))                    # largura entre 8 e 60
        aba.freeze_panes(1, 0)                                             # congela o cabeçalho
        aba.autofilter(0, 0, len(tabela), len(tabela.columns) - 1)         # liga o filtro
    else:
        formatar_excel(escritor, nome_aba)


def escritor_excel(caminho):
    """Abre um arquivo Excel para gravação com o motor disponível."""
    return pd.ExcelWriter(caminho, engine=MOTOR_EXCEL)


log.info("Funções auxiliares definidas!")

## 3. Carregar entradas
A rais classificada tem **uma linha por registro** (todos os anos, todos os CEPs). Os pontos vêm da camada `ceps_pontos` (um ponto por CEP), que está em metros (EPSG:31983) e passa para graus (EPSG:4674).

In [ ]:
log.info("Carregando a rais classificada...")
rais_cls = pd.read_parquet(ARQ_RAIS_CLASS)
rais_cls["linha_rais"] = np.arange(len(rais_cls), dtype="int64")           # mesma numeração usada na etapa 06 (rastreio)
for col in ["ano", "cep_chave", "classe_cep", "motivo_cep", "qtd_vinc_ativos"]:
    assert col in rais_cls.columns, f"Coluna '{col}' não existe na rais classificada (rode a Etapa 05 atualizada)"
N_RAIS = len(rais_cls)
log.info("rais classificada: %d linhas | anos: %s", N_RAIS, sorted(rais_cls["ano"].unique().tolist()))

log.info("Carregando os pontos dos CEPs...")
pontos_cep = pyogrio.read_dataframe(ARQ_PONTOS_CEP, layer="ceps_pontos",
                                    columns=["cep", "processo", "confianca", "ponto_metodo", "n_partes"])
assert pontos_cep.crs is not None and pontos_cep.crs.to_string() == CRS_METRICO, f"CRS inesperado: {pontos_cep.crs}"
pontos_cep = pontos_cep.to_crs(CRS_GRAUS)                                  # metros -> graus (a entrega é em EPSG:4674)
validar_tabela("Pontos dos CEPs", pontos_cep, chave=["cep"], obrigatorias=["cep"])

log.info("Carregando a rais consolidada (só para conferir o total)...")
rais_orig = pd.read_parquet(ARQ_RAIS_CONSOL, columns=["ano", "qtd_vinc_ativos"])
log.info("rais consolidada: %d linhas | vínculos ativos: %d", len(rais_orig), int(pd.to_numeric(rais_orig["qtd_vinc_ativos"], errors="coerce").sum()))
log.info("Entradas carregadas!")

## 4. Montar a tabela final
Cada registro herda o ponto do seu CEP (se houver). Quem **não** tem ponto continua na tabela, com geometria, `lat` e `lon` vazios e `situacao_espacial = nao_localizavel`.

In [ ]:
log.info("Montando a tabela final (todos os registros)...")
pts = pd.DataFrame({
    "cep_chave": pontos_cep["cep"].values,
    "processo_cep": pontos_cep["processo"].values,
    "confianca_cep": pontos_cep["confianca"].values,
    "ponto_metodo": pontos_cep["ponto_metodo"].values,
    "n_partes": pontos_cep["n_partes"].values,
    "lat": pontos_cep.geometry.y.round(6).values,                         # latitude em graus, 6 casas (cerca de 10 cm)
    "lon": pontos_cep.geometry.x.round(6).values,                         # longitude em graus
    "geometry": pontos_cep.geometry.values})
tab = rais_cls.merge(pts, on="cep_chave", how="left", validate="many_to_one")   # um registro continua sendo uma linha
assert len(tab) == N_RAIS, "A junção com os pontos mudou o número de linhas da rais!"

tem_ponto = tab["geometry"].notna()                                        # quem recebeu ponto do CEP
classe_com_ponto = tab["classe_cep"].isin(CLASSES_PONTO)                   # quem deveria ter ponto pela classe
if int((tem_ponto != classe_com_ponto).sum()):
    log.error("Inconsistência: %d registros com classe/ponto divergentes (rode as etapas 05 e 06 de novo).", int((tem_ponto != classe_com_ponto).sum()))
tab["situacao_espacial"] = np.where(tem_ponto, "espacializado", "nao_localizavel")
tab["geometry"] = tab["geometry"].where(tem_ponto, None)                   # sem ponto = sem geometria

# --- Colunas: originais da rais + as do projeto -------------------------------------------------------------------------
fora = ["cep_chave", "espacializavel", "linha_rais", "classe_cep", "motivo_cep"] + list(COLUNAS_EXCLUIR)
colunas_rais = [c for c in rais_cls.columns if c not in fora]               # colunas originais da rais, na ordem original
colunas_projeto = ["situacao_espacial", "classe_cep", "motivo_cep", "processo_cep", "confianca_cep", "ponto_metodo", "n_partes", "lat", "lon"]
colunas_final = ["linha_rais"] + colunas_rais + colunas_projeto
tab = tab.sort_values(["ano", "linha_rais"]).reset_index(drop=True)         # ordem estável: ano e posição original
log.info("Colunas da entrega (%d): %d da rais + %d do projeto + linha_rais", len(colunas_final), len(colunas_rais), len(colunas_projeto))

log.info("Registros por situação espacial:\n%s", tab["situacao_espacial"].value_counts().to_string())
log.info("Por confiança (registros):\n%s", tab["confianca_cep"].fillna("(sem ponto)").value_counts().to_string())
sem_confianca = int((tem_ponto & tab["confianca_cep"].isna()).sum())
(log.info if sem_confianca == 0 else log.warning)("Registros com ponto e sem confiança informada: %d", sem_confianca)
validar_tabela("Tabela final da rais", tab, chave=["linha_rais"], obrigatorias=["linha_rais", "ano", "situacao_espacial"], linhas_entrada=N_RAIS)
log.info("Tabela final montada!")

## 5. GeoPackage
Uma camada só. Os registros `nao_localizavel` entram **sem geometria** (o GeoPackage aceita isso; no QGIS aparecem só na tabela de atributos). Depois de gravar, o arquivo é reaberto e conferido.

In [ ]:
log.info("Gravando o GeoPackage...")
t0 = time.time()
gdf = gpd.GeoDataFrame(tab[colunas_final + ["geometry"]], geometry="geometry", crs=CRS_GRAUS)
if ARQ_GPKG.exists():
    ARQ_GPKG.unlink()                                                      # recomeça do zero (arquivo derivado, regerável)
para_gpkg(gdf).to_file(ARQ_GPKG, layer=CAMADA_GPKG, driver="GPKG", engine="pyogrio")
log.info("GeoPackage gravado: %s (%.1f MB) em %.0f s", ARQ_GPKG.name, ARQ_GPKG.stat().st_size / 1e6, time.time() - t0)

log.info("Reabrindo o GeoPackage para conferir...")
camadas = [c[0] for c in pyogrio.list_layers(ARQ_GPKG)]
rev = pyogrio.read_dataframe(ARQ_GPKG, layer=CAMADA_GPKG)
n_esp = int(tem_ponto.sum())
conferencias = [
    ("Camadas no arquivo (esperado: só " + CAMADA_GPKG + ")", camadas == [CAMADA_GPKG]),
    ("Linhas (esperado %d)" % N_RAIS, len(rev) == N_RAIS),
    ("Com geometria (esperado %d)" % n_esp, int(rev.geometry.notna().sum()) == n_esp),
    ("Sem geometria (esperado %d)" % (N_RAIS - n_esp), int(rev.geometry.isna().sum()) == N_RAIS - n_esp),
    ("Mesma ordem de registros (linha_rais)", bool((rev["linha_rais"].values == tab["linha_rais"].values).all())),
    ("Colunas iguais às esperadas", list(rev.columns) == colunas_final + ["geometry"]),
    ("CRS %s" % CRS_GRAUS, rev.crs is not None and rev.crs.to_string() == CRS_GRAUS),
    ("lat/lon vazios exatamente nos sem geometria", bool((rev["lat"].isna() == rev.geometry.isna()).all() and (rev["lon"].isna() == rev.geometry.isna()).all())),
    ("Ponto igual a lon/lat (tolerância 1e-6 grau)",
     bool(((rev.geometry.x - rev["lon"]).abs().max() < 1e-6) and ((rev.geometry.y - rev["lat"]).abs().max() < 1e-6)) if n_esp else True),
    ("Vínculos ativos = total da rais", int(pd.to_numeric(rev["qtd_vinc_ativos"], errors="coerce").sum()) == int(pd.to_numeric(rais_cls["qtd_vinc_ativos"], errors="coerce").sum())),
]
for _nome, _ok in conferencias:
    (log.info if _ok else log.error)("  [%s] %s", "OK" if _ok else "ERRO", _nome)
del rev                                                                    # libera memória
log.info("GeoPackage conferido! %s | camada %s | %d registros (%d com ponto, %d sem)", ARQ_GPKG.name, CAMADA_GPKG, N_RAIS, n_esp, N_RAIS - n_esp)

## 6. Excel dos registros (uma aba por ano)
Mesmas linhas e colunas do GeoPackage, **sem a geometria** (`lat` e `lon` ficam, vazias nos `nao_localizavel`). Cada ano tem bem menos que o limite de linhas do Excel; a aba `legenda` explica as colunas do projeto.

In [ ]:
log.info("Preparando o Excel dos registros...")
excel_tab = pd.DataFrame(tab[colunas_final])                               # sem a coluna geometry
excel_tab, n_mudadas = limpar_texto_excel(excel_tab)
log.info("Células com caracteres inválidos para o Excel (removidos só da planilha): %d", n_mudadas)
anos = sorted(excel_tab["ano"].unique().tolist())

LEGENDA_PROJETO = {
    "linha_rais": "Posição do registro na RAIS consolidada (rastreio entre arquivos).",
    "situacao_espacial": "espacializado = o CEP do registro recebeu um ponto; nao_localizavel = sem ponto (CEP vazio, inválido, de outro município, geral do município ou sem endereço localizável). Registro de CEP de outro município continua sendo de São José dos Campos.",
    "classe_cep": "Classe do CEP: A (via curta), B (via longa ou várias vias), G (localizado pelo endereço), P (pontos do CNEFE no CEP) têm ponto; C, D e E não têm.",
    "motivo_cep": "Coluna técnica: motivo da classe do CEP (ex.: prefixo_fora_de_12, cep_geral_sufixo_000, ausente_nos_correios).",
    "processo_cep": "Como o CEP foi ligado ao ponto: correios_geosanja, cnefe_segmento, manual, localizacao_endereco ou cnefe_pontos_cep.",
    "confianca_cep": "Confiança do ponto do CEP: alta, media, baixa ou manual. Vazia nos nao_localizavel.",
    "ponto_metodo": "Método que gerou o ponto (ex.: meio_da_linha, centroide_ajustado, endereco:cnefe_rua).",
    "n_partes": "Número de partes (trechos) do CEP usadas no ponto.",
    "lat": "Latitude do ponto do CEP em graus (EPSG:4674). Vazia nos nao_localizavel.",
    "lon": "Longitude do ponto do CEP em graus (EPSG:4674). Vazia nos nao_localizavel.",
}
legenda = pd.DataFrame({"coluna": colunas_final,
                        "descricao": [LEGENDA_PROJETO.get(c, "Coluna original da RAIS (ver docs/fontes.md).") for c in colunas_final]})

log.info("Gravando o Excel: %s (%d abas de ano)...", ARQ_XLSX.name, len(anos))
t0 = time.time()
if ARQ_XLSX.exists():
    ARQ_XLSX.unlink()
with escritor_excel(ARQ_XLSX) as escritor:
    for ano in anos:                                                       # uma aba por ano
        parte = excel_tab[excel_tab["ano"] == ano]
        log.info("  Aba %s: gravando %d linhas...", ano, len(parte))
        gravar_aba(escritor, str(ano), parte)
        log.info("  Aba %s gravada!", ano)
    gravar_aba(escritor, "legenda", legenda)
    log.info("Fechando o arquivo (é nesta hora que ele é montado, pode demorar)...")
log.info("Excel gravado: %s (%.1f MB) em %.0f s", ARQ_XLSX.name, ARQ_XLSX.stat().st_size / 1e6, time.time() - t0)

log.info("Reabrindo o Excel para conferir (só conta linhas e lê o cabeçalho)...")
from openpyxl import load_workbook
wb = load_workbook(ARQ_XLSX, read_only=True)
abas_esperadas = [str(a) for a in anos] + ["legenda"]
ok_abas = wb.sheetnames == abas_esperadas
(log.info if ok_abas else log.error)("  [%s] Abas: %s", "OK" if ok_abas else "ERRO", wb.sheetnames)
total_linhas = 0
for ano in anos:
    ws = wb[str(ano)]
    n_aba = ws.max_row - 1                                                 # menos o cabeçalho
    cab = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    esperado = int((excel_tab["ano"] == ano).sum())
    ok = n_aba == esperado and cab == colunas_final
    total_linhas += n_aba
    (log.info if ok else log.error)("  [%s] Aba %s: %d linhas (esperado %d) | cabeçalho igual: %s", "OK" if ok else "ERRO", ano, n_aba, esperado, cab == colunas_final)
wb.close()
(log.info if total_linhas == N_RAIS else log.error)("  [%s] Soma das abas de ano: %d (esperado %d)", "OK" if total_linhas == N_RAIS else "ERRO", total_linhas, N_RAIS)
log.info("Excel dos registros conferido!")

## 7. Saldo anual de vínculos ativos
**Saldo = soma de `qtd_vinc_ativos` (vínculos ativos em 31/12) de todos os registros da RAIS no ano**, tenham ou não ponto. A variação é a diferença para o ano anterior. A aba `por_ano` mostra também quanto do saldo é `espacializado` e quanto é `nao_localizavel`; a aba `por_ano_confianca` abre o saldo por confiança do ponto.

In [ ]:
log.info("Calculando o saldo anual de vínculos ativos...")
saldo_base = tab[["ano", "situacao_espacial", "confianca_cep", "qtd_vinc_ativos"] +
                 [c for c in ["qtd_vinc_clt", "qtd_vinc_estatutarios"] if c in tab.columns]].copy()
for col in [c for c in saldo_base.columns if c.startswith("qtd_")]:
    saldo_base[col] = pd.to_numeric(saldo_base[col], errors="coerce").fillna(0)    # vazio conta como zero
saldo_base["confianca"] = saldo_base["confianca_cep"].where(saldo_base["situacao_espacial"] == "espacializado", "nao_localizavel")
saldo_base["confianca"] = saldo_base["confianca"].fillna("sem_confianca")          # ponto sem confiança informada (não deve ocorrer)

# --- Aba por_ano (uma linha por ano) -----------------------------------------------------------------------------------------
por_ano = saldo_base.groupby("ano").agg(
    n_estabelecimentos=("ano", "size"), vinc_ativos=("qtd_vinc_ativos", "sum"),
    **({"vinc_clt": ("qtd_vinc_clt", "sum")} if "qtd_vinc_clt" in saldo_base else {}),
    **({"vinc_estatutarios": ("qtd_vinc_estatutarios", "sum")} if "qtd_vinc_estatutarios" in saldo_base else {}))
por_ano["variacao_vinc_ativos"] = por_ano["vinc_ativos"].diff()                      # saldo do ano menos o do ano anterior
por_ano["variacao_pct_vinc_ativos"] = (por_ano["vinc_ativos"].pct_change() * 100).round(2)
por_ano["variacao_pct_estabelecimentos"] = (por_ano["n_estabelecimentos"].pct_change() * 100).round(2)
for sit, sufixo in [("espacializado", "espacializados"), ("nao_localizavel", "nao_localizaveis")]:
    sub = saldo_base[saldo_base["situacao_espacial"] == sit].groupby("ano").agg(n=("ano", "size"), v=("qtd_vinc_ativos", "sum"))
    por_ano[f"n_estab_{sufixo}"] = sub["n"].reindex(por_ano.index).fillna(0).astype("int64")
    por_ano[f"vinc_ativos_{sufixo}"] = sub["v"].reindex(por_ano.index).fillna(0)
    por_ano[f"pct_vinc_{sufixo}"] = (por_ano[f"vinc_ativos_{sufixo}"] / por_ano["vinc_ativos"] * 100).round(2)
por_ano = por_ano.reset_index()
for col in por_ano.columns:                                                        # somas inteiras viram inteiro (sem ".0")
    if col.startswith(("vinc_", "n_")) and not col.startswith("pct"):
        por_ano[col] = por_ano[col].astype("int64")

# --- Aba por_ano_confianca (uma linha por ano e confiança) ---------------------------------------------------------------
conf = saldo_base.groupby(["ano", "confianca"]).agg(n_estabelecimentos=("ano", "size"), vinc_ativos=("qtd_vinc_ativos", "sum"))
niveis = [c for c in ORDEM_CONFIANCA if c in set(saldo_base["confianca"])] + sorted(set(saldo_base["confianca"]) - set(ORDEM_CONFIANCA))
conf = conf.reindex(pd.MultiIndex.from_product([sorted(saldo_base["ano"].unique()), niveis], names=["ano", "confianca"]), fill_value=0).reset_index()
conf["pct_vinc_ativos"] = (conf["vinc_ativos"] / conf.groupby("ano")["vinc_ativos"].transform("sum") * 100).round(2)
conf["vinc_ativos"] = conf["vinc_ativos"].astype("int64")

# --- Conferências -------------------------------------------------------------------------------------------------------------------
log.info("Conferindo o saldo...")
orig_v = pd.to_numeric(rais_orig["qtd_vinc_ativos"], errors="coerce").fillna(0).groupby(rais_orig["ano"]).sum()
orig_n = rais_orig.groupby("ano").size()
for _, lin in por_ano.iterrows():
    ano = int(lin["ano"])
    ok_orig = int(lin["vinc_ativos"]) == int(orig_v.get(ano, -1)) and int(lin["n_estabelecimentos"]) == int(orig_n.get(ano, -1))
    ok_soma = int(lin["vinc_ativos_espacializados"] + lin["vinc_ativos_nao_localizaveis"]) == int(lin["vinc_ativos"]) and \
              int(lin["n_estab_espacializados"] + lin["n_estab_nao_localizaveis"]) == int(lin["n_estabelecimentos"])
    ok_conf = int(conf.loc[conf["ano"] == ano, "vinc_ativos"].sum()) == int(lin["vinc_ativos"])
    nivel = log.info if (ok_orig and ok_soma and ok_conf) else log.error
    nivel("  [%s] %d: %d estab. | %d vínculos | = rais consolidada: %s | espacializado + nao_localizavel = total: %s | soma por confiança = total: %s",
          "OK" if (ok_orig and ok_soma and ok_conf) else "ERRO", ano, lin["n_estabelecimentos"], lin["vinc_ativos"],
          "sim" if ok_orig else "NÃO", "sim" if ok_soma else "NÃO", "sim" if ok_conf else "NÃO")
for _, lin in por_ano.dropna(subset=["variacao_pct_estabelecimentos"]).iterrows():
    if abs(lin["variacao_pct_estabelecimentos"]) > LIMITE_ALERTA * 100:
        log.warning("  Atenção: o número de estabelecimentos de %d variou %.1f%% em relação ao ano anterior (limite %.0f%%). Confira a RAIS desse ano.",
                    int(lin["ano"]), lin["variacao_pct_estabelecimentos"], LIMITE_ALERTA * 100)
tot_ok = int(por_ano["vinc_ativos"].sum()) == int(orig_v.sum()) and int(por_ano["n_estabelecimentos"].sum()) == len(rais_orig)
(log.info if tot_ok else log.error)("  [%s] Total de todos os anos: %d estabelecimentos | %d vínculos (rais consolidada: %d | %d)",
                                    "OK" if tot_ok else "ERRO", int(por_ano["n_estabelecimentos"].sum()), int(por_ano["vinc_ativos"].sum()), len(rais_orig), int(orig_v.sum()))
validar_tabela("Saldo por ano", por_ano, chave=["ano"], obrigatorias=["ano", "vinc_ativos"])

# --- Excel do saldo ------------------------------------------------------------------------------------------------------------------
legenda_saldo = pd.DataFrame([
    ("ano", "Ano da RAIS."),
    ("n_estabelecimentos", "Registros (estabelecimentos) da RAIS no ano, com ou sem ponto."),
    ("vinc_ativos", "SALDO: soma de qtd_vinc_ativos (vínculos ativos em 31/12) de todos os registros do ano."),
    ("vinc_clt / vinc_estatutarios", "Soma de qtd_vinc_clt e qtd_vinc_estatutarios no ano."),
    ("variacao_vinc_ativos / variacao_pct_vinc_ativos", "Diferença (e % de variação) do saldo em relação ao ano anterior. Vazia no primeiro ano."),
    ("variacao_pct_estabelecimentos", "% de variação do número de registros em relação ao ano anterior."),
    ("n_estab_espacializados / vinc_ativos_espacializados / pct_vinc_espacializados", "Registros e vínculos cujo CEP recebeu ponto, e % do saldo."),
    ("n_estab_nao_localizaveis / vinc_ativos_nao_localizaveis / pct_vinc_nao_localizaveis", "Registros e vínculos sem ponto (CEP vazio, inválido, de outro município, geral do município ou sem endereço localizável), e % do saldo. Continuam contados no saldo do município."),
    ("confianca (aba por_ano_confianca)", "Confiança do ponto do CEP (alta, media, baixa, manual) ou nao_localizavel; vinc_ativos e pct_vinc_ativos (% do saldo do ano)."),
], columns=["coluna", "descricao"])
log.info("Gravando o Excel do saldo: %s...", ARQ_SALDO.name)
if ARQ_SALDO.exists():
    ARQ_SALDO.unlink()
with escritor_excel(ARQ_SALDO) as escritor:
    gravar_aba(escritor, "por_ano", por_ano)
    gravar_aba(escritor, "por_ano_confianca", conf)
    gravar_aba(escritor, "legenda", legenda_saldo)
rel = pd.read_excel(ARQ_SALDO, sheet_name=None)                            # reabre para conferir
ok_rel = list(rel) == ["por_ano", "por_ano_confianca", "legenda"] and rel["por_ano"]["vinc_ativos"].tolist() == por_ano["vinc_ativos"].tolist() \
    and rel["por_ano_confianca"]["vinc_ativos"].tolist() == conf["vinc_ativos"].tolist()
(log.info if ok_rel else log.error)("  [%s] Excel do saldo reaberto: abas %s | números iguais às tabelas", "OK" if ok_rel else "ERRO", list(rel))
log.info("Saldo anual pronto!")

## 8. Resumo

In [ ]:
log.info("Resumo final...")
log.info("Registros da rais: %d | espacializados: %d (%.1f%%) | nao_localizavel: %d (%.1f%%)",
         N_RAIS, n_esp, n_esp / N_RAIS * 100, N_RAIS - n_esp, (N_RAIS - n_esp) / N_RAIS * 100)
log.info("Saldo de vínculos ativos por ano:\n%s", por_ano[["ano", "n_estabelecimentos", "vinc_ativos", "variacao_vinc_ativos",
                                                          "pct_vinc_espacializados", "pct_vinc_nao_localizaveis"]].to_string(index=False))
log.info("Arquivos gerados em %s:", ARQ_GPKG.parent)
for _arq in [ARQ_GPKG, ARQ_XLSX, ARQ_SALDO]:
    log.info("  %s (%.1f MB)", _arq.name, _arq.stat().st_size / 1e6)
log.info("Etapa 07 concluída!")